# 🛡️ Quantum IDS: Advanced EDA & Unified Intelligence

This notebook provides an interactive environment to explore the **NSL-KDD** and **CIC-IDS 2017** datasets, visualize threat distributions, and train the unified machine learning model using over **2.4 Million network packets**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Setting aesthetic style
plt.style.use('dark_background')
sns.set_palette("viridis")
print("✅ Environment Setup Complete")

✅ Environment Setup Complete


## 1. Load Datasets
We use the merged CSV files created during the backend phase. 
> **Note**: We will sample the data for plotting to keep the notebook responsive.

In [2]:
nsl_path = "../data/processed/nslkdd_merged.csv"
cic_path = "../data/processed/cicids_merged.csv"

print("📊 Loading NSL-KDD...")
nsl_df = pd.read_csv(nsl_path)
print(f"   Shape: {nsl_df.shape}")

print("📊 Loading CIC-IDS 2017 (This may take a moment due to 2.3M rows)...")
cic_df = pd.read_csv(cic_path)
print(f"   Shape: {cic_df.shape}")

📊 Loading NSL-KDD...
   Shape: (148517, 43)
📊 Loading CIC-IDS 2017 (This may take a moment due to 2.3M rows)...
   Shape: (2313810, 78)


## 2. Interactive EDA: Class Distributions
Let's see how many attacks vs benign packets exist in our world.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(x=nsl_df['label'], ax=ax[0], palette="magma")
ax[0].set_title("NSL-KDD Class Distribution")
ax[0].tick_params(axis='x', rotation=45)

sns.countplot(x=cic_df['label'], ax=ax[1], palette="viridis")
ax[1].set_title("CIC-IDS 2017 Class Distribution")
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Feature Correlations
Visualizing how network metrics relate to each other.

In [ ]:
plt.figure(figsize=(12, 8))
nsl_numeric = nsl_df.select_dtypes(include=[np.number])
sns.heatmap(nsl_numeric.corr(), cmap="coolwarm", annot=False)
plt.title("NSL-KDD Correlation Heatmap")
plt.show()

## 4. Unified Training Pipeline
We map the different features into a single **Super-Intelligence** vector.

In [ ]:
def map_unified_features(nsl, cic):
    # Standardize Labels
    nsl['label_std'] = nsl['label'].apply(lambda x: 0 if x == 'normal' else 1)
    cic['label_std'] = cic['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)
    
    # Common Feature Mapping
    common_nsl = nsl[['duration', 'protocol_type', 'src_bytes', 'dst_bytes', 'count', 'srv_count', 'label_std']].copy()
    
    common_cic = pd.DataFrame()
    common_cic['duration'] = cic['Flow Duration'] / 1000000.0
    common_cic['protocol_type'] = cic['Protocol'].map({6: 'tcp', 17: 'udp', 1: 'icmp'}).fillna('tcp')
    common_cic['src_bytes'] = cic['Fwd Packets Length Total']
    common_cic['dst_bytes'] = cic['Bwd Packets Length Total']
    common_cic['count'] = cic['Total Fwd Packets']
    common_cic['srv_count'] = cic['Total Backward Packets']
    common_cic['label_std'] = cic['label_std']
    
    return pd.concat([common_nsl, common_cic], ignore_index=True)

print("🔗 Merging into Unified Super-Dataset...")
super_df = map_unified_features(nsl_df, cic_df)
print(f"✨ Super-Dataset Ready: {super_df.shape}")

## 5. Model Evaluation
Training the Random Forest model on the unified intelligence.

In [ ]:
X = pd.get_dummies(super_df.drop('label_std', axis=1))
y = super_df['label_std']

# Sample for training in notebook speed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧠 Training Unified Random Forest (on sample)...")
rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("\n📈 Classification Report:")
print(classification_report(y_test, y_pred))